# 🏥 NetraAI (SIH26038) — IEEE IDRiD Complete Training Pipeline
### Indian Diabetic Retinopathy Image Dataset (IEEE Dataport)

This dedicated Google Colab notebook trains **two specialized clinical AI models** on the IEEE IDRiD dataset in **~10–12 minutes** using a free **NVIDIA T4 GPU (16GB VRAM)**:

1. **Task 1 (Part A: Segmentation)**: Deep U-Net for pixel-level segmentation of **Microaneurysms (MAs), Haemorrhages (HEs), Hard Exudates (EXs), and Soft Exudates (SEs)** $\rightarrow$ Outputs `unet_lesions.pt`
2. **Task 2 (Part B: Disease Grading)**: EfficientNet-B3 Dual-Head Model for **5-Class DR Severity (ICDR 0–4)** and **3-Class Diabetic Macular Edema (DME Risk 0–2)** $\rightarrow$ Outputs `idrid_grading_efficientnet_b3.pt`

---

## 1. Verify GPU Activation (T4 GPU 16GB)
Ensure **Runtime > Change runtime type > T4 GPU** is selected in Google Colab.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Install Required Deep Learning & Vision Libraries

In [ ]:
!pip install timm segmentation-models-pytorch opencv-python-headless scikit-learn pandas pillow matplotlib tifffile -q
import os
os.makedirs("/content/checkpoints", exist_ok=True)
print("✓ Dependencies installed and checkpoint directory created!")

## 3. Mount Google Drive or Extract IDRiD Dataset
Mount your Google Drive where your `IDRID(IEEE)` folder or zip is stored.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Search for IDRiD path in Google Drive
possible_paths = [
    "/content/drive/MyDrive/Datasests/IDRID(IEEE)",
    "/content/drive/MyDrive/Datasets/IDRID(IEEE)",
    "/content/drive/MyDrive/IDRID(IEEE)",
    "/content/drive/MyDrive/IDRiD",
    "/content/IDRID(IEEE)",
    "/content/dataset/IDRID(IEEE)"
]

IDRID_DIR = None
for p in possible_paths:
    if os.path.exists(p):
        IDRID_DIR = p
        break

# If not found in drive, check if a zip file exists to unzip
if IDRID_DIR is None:
    zip_candidates = [
        "/content/drive/MyDrive/IDRID.zip",
        "/content/drive/MyDrive/Datasests/IDRID(IEEE).zip",
        "/content/IDRID.zip"
    ]
    for z in zip_candidates:
        if os.path.exists(z):
            print(f"Unzipping {z}...")
            !unzip -q "{z}" -d /content/IDRID_extracted
            IDRID_DIR = "/content/IDRID_extracted"
            break

if IDRID_DIR is not None:
    print(f"✅ IDRiD Dataset directory located: {IDRID_DIR}")
    print("Contents:", os.listdir(IDRID_DIR))
else:
    print("⚠️ IDRiD directory not automatically detected. Please set IDRID_DIR manually below.")
    IDRID_DIR = "/content/drive/MyDrive/Datasests/IDRID(IEEE)"

## 4. PART A: Train Multi-Lesion Segmentation U-Net
### Targets: Microaneurysms (MAs), Haemorrhages (HEs), Hard Exudates (EXs), Soft Exudates (SEs)
- **Architecture**: U-Net with pre-trained ResNet-34 encoder.
- **Loss Function**: Combined Binary Cross-Entropy + Soft Dice Loss (`DiceBCELoss`).
- **Resolution**: 512×512 with Ben Graham local contrast enhancement.
- **Estimated GPU Time**: ~4–5 minutes (25 epochs on 81 annotated scans).

In [ ]:
import cv2
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import tifffile as tiff

# 1. Preprocessing Utilities
def crop_to_circle_mask(image, tol=7):
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        mask = gray > tol
        if mask.any():
            img1 = image[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = image[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = image[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            return np.stack([img1, img2, img3], axis=-1)
    return image

def apply_ben_graham(image, target_size=512):
    resized = cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(resized, (0, 0), target_size / 30.0)
    enhanced = cv2.addWeighted(resized, 4.0, blurred, -4.0, 128)
    mask = np.zeros((target_size, target_size), dtype=np.uint8)
    cv2.circle(mask, (target_size // 2, target_size // 2), int(target_size * 0.48), 255, -1)
    return cv2.bitwise_and(enhanced, enhanced, mask=mask)

# 2. Segmentation Dataset Loader
class IDRiDSegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_base_dir, is_training=True, target_size=512):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.[jJ][pP][gG]")) + glob.glob(os.path.join(img_dir, "*.[tT][iI][fF]*")))
        self.mask_base_dir = mask_base_dir
        self.is_training = is_training
        self.target_size = target_size
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1).astype(np.float32)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1).astype(np.float32)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_p = self.img_paths[idx]
        base_id = os.path.splitext(os.path.basename(img_p))[0]  # e.g., IDRiD_01

        img = cv2.imread(img_p)
        if img is None:
            img = tiff.imread(img_p)
        img_proc = apply_ben_graham(crop_to_circle_mask(img), target_size=self.target_size)
        rgb = cv2.cvtColor(img_proc, cv2.COLOR_BGR2RGB)

        # Combine multi-lesion masks (MAs, HEs, Hard EXs, Soft SEs)
        combined_mask = np.zeros((self.target_size, self.target_size), dtype=np.float32)
        lesion_dirs = ["1. Microaneurysms", "2. Haemorrhages", "3. Hard Exudates", "4. Soft Exudates"]
        
        for l_dir in lesion_dirs:
            search_p = os.path.join(self.mask_base_dir, l_dir, f"{base_id}*.*")
            matches = glob.glob(search_p)
            if matches:
                m = cv2.imread(matches[0], cv2.IMREAD_GRAYSCALE)
                if m is None:
                    m = tiff.imread(matches[0])
                m_resized = cv2.resize(m.astype(np.float32), (self.target_size, self.target_size), interpolation=cv2.INTER_NEAREST)
                combined_mask = np.maximum(combined_mask, (m_resized > 10).astype(np.float32))

        # Data augmentation
        tensor_img = rgb.astype(np.float32) / 255.0
        if self.is_training:
            if np.random.rand() > 0.5:
                tensor_img = np.fliplr(tensor_img).copy()
                combined_mask = np.fliplr(combined_mask).copy()
            if np.random.rand() > 0.5:
                tensor_img = np.flipud(tensor_img).copy()
                combined_mask = np.flipud(combined_mask).copy()

        tensor_img = np.transpose(tensor_img, (2, 0, 1))
        tensor_img = (tensor_img - self.mean) / self.std
        tensor_mask = np.expand_dims(combined_mask, axis=0)

        return torch.tensor(tensor_img, dtype=torch.float32), torch.tensor(tensor_mask, dtype=torch.float32)

# Set Segmentation Paths
train_img_dir = os.path.join(IDRID_DIR, "A. Segmentation/1. Original Images/a. Training Set")
train_mask_dir = os.path.join(IDRID_DIR, "A. Segmentation/2. All Segmentation Groundtruths/a. Training Set")
test_img_dir = os.path.join(IDRID_DIR, "A. Segmentation/1. Original Images/b. Testing Set")
test_mask_dir = os.path.join(IDRID_DIR, "A. Segmentation/2. All Segmentation Groundtruths/b. Testing Set")

train_seg_ds = IDRiDSegmentationDataset(train_img_dir, train_mask_dir, is_training=True)
test_seg_ds = IDRiDSegmentationDataset(test_img_dir, test_mask_dir, is_training=False)

train_seg_loader = DataLoader(train_seg_ds, batch_size=8, shuffle=True, num_workers=2)
test_seg_loader = DataLoader(test_seg_ds, batch_size=8, shuffle=False, num_workers=2)

print(f"Segmentation Samples: {len(train_seg_ds)} Training Images | {len(test_seg_ds)} Testing Images")

### Train U-Net Architecture on GPU

In [ ]:
# Loss function: Combined BCE + Dice Loss
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceBCELoss, self).__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (probs_flat * targets_flat).sum()
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        return bce_loss + dice_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create U-Net with ResNet-34 encoder
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

seg_criterion = DiceBCELoss()
seg_optimizer = optim.AdamW(unet_model.parameters(), lr=5e-4, weight_decay=1e-4)
seg_scheduler = optim.lr_scheduler.CosineAnnealingLR(seg_optimizer, T_max=25, eta_min=1e-6)
seg_scaler = torch.amp.GradScaler('cuda')

best_dice = 0.0
print(f"\n--- [Task 1] Training Multi-Lesion U-Net on {device} (25 Epochs) ---")

for epoch in range(1, 26):
    unet_model.train()
    train_loss = 0.0
    for imgs, masks in train_seg_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        seg_optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = unet_model(imgs)
            loss = seg_criterion(logits, masks)
        seg_scaler.scale(loss).backward()
        seg_scaler.step(seg_optimizer)
        seg_scaler.update()
        train_loss += loss.item() * imgs.size(0)
    seg_scheduler.step()

    # Validation / Test Evaluation
    unet_model.eval()
    val_loss, total_intersection, total_union = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in test_seg_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.amp.autocast('cuda'):
                logits = unet_model(imgs)
                loss = seg_criterion(logits, masks)
            val_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            total_intersection += (preds * masks).sum().item()
            total_union += (preds + masks).sum().item()

    dice_score = (2.0 * total_intersection + 1e-5) / (total_union + 1e-5)
    avg_train_loss = train_loss / len(train_seg_ds)
    avg_val_loss = val_loss / len(test_seg_ds)

    print(f"Epoch [{epoch:02d}/25] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Dice Score: {dice_score:.4f}")

    if dice_score > best_dice:
        best_dice = dice_score
        torch.save({
            'epoch': epoch,
            'model_state_dict': unet_model.state_dict(),
            'best_dice': best_dice,
            'architecture': 'unet_resnet34'
        }, "/content/checkpoints/unet_lesions.pt")
        print(f"  >>> ⭐ Saved best lesion U-Net: /content/checkpoints/unet_lesions.pt (Dice: {best_dice:.4f})")

print(f"\n✓ [Task 1 Complete] Best Lesion Segmentation Dice Score: {best_dice:.4f}")

## 5. PART B: Train Dual-Head DR & DME Severity Model
### Multi-Task Clinical Grading on IDRiD Disease Grading Ground Truths
- **Head 1**: 5-Class Retinopathy Severity (ICDR Grade 0 to 4)
- **Head 2**: 3-Class Diabetic Macular Edema Risk (DME Grade 0 to 2)
- **Architecture**: EfficientNet-B3 with Dual Multi-Layer Classification Heads
- **Estimated GPU Time**: ~5–6 minutes (20 epochs on 413 training & 103 test scans).

In [ ]:
import pandas as pd
import torchvision.models as models
from sklearn.metrics import cohen_kappa_score, classification_report, accuracy_score

# 1. Load CSV Ground Truths
train_csv_p = os.path.join(IDRID_DIR, "B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv")
test_csv_p = os.path.join(IDRID_DIR, "B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv")
train_img_dir_b = os.path.join(IDRID_DIR, "B. Disease Grading/1. Original Images/a. Training Set")
test_img_dir_b = os.path.join(IDRID_DIR, "B. Disease Grading/1. Original Images/b. Testing Set")

df_train = pd.read_csv(train_csv_p)
df_test = pd.read_csv(test_csv_p)

# Clean column names
df_train.columns = [c.strip() for c in df_train.columns]
df_test.columns = [c.strip() for c in df_test.columns]

print("Training CSV Columns:", df_train.columns.tolist())
print(f"Training Records: {len(df_train)} | Testing Records: {len(df_test)}")
print("DR Grade distribution in Train:", df_train['Retinopathy grade'].value_counts().to_dict())

# 2. Disease Grading Dataset
class IDRiDGradingDataset(Dataset):
    def __init__(self, df, img_dir, is_training=True, target_size=512):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.is_training = is_training
        self.target_size = target_size
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1).astype(np.float32)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1).astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['Image name']).strip()
        # Check for .jpg / .tif extensions
        img_p = os.path.join(self.img_dir, f"{img_name}.jpg")
        if not os.path.exists(img_p):
            img_p = os.path.join(self.img_dir, f"{img_name}.tif")

        img = cv2.imread(img_p)
        if img is None:
            img = np.zeros((512, 512, 3), dtype=np.uint8)

        proc_img = apply_ben_graham(crop_to_circle_mask(img), target_size=self.target_size)
        rgb = cv2.cvtColor(proc_img, cv2.COLOR_BGR2RGB)
        tensor = rgb.astype(np.float32) / 255.0

        if self.is_training:
            if np.random.rand() > 0.5: tensor = np.fliplr(tensor).copy()
            if np.random.rand() > 0.5: tensor = np.flipud(tensor).copy()

        tensor = np.transpose(tensor, (2, 0, 1))
        tensor = (tensor - self.mean) / self.std

        dr_label = int(row['Retinopathy grade'])
        # DME column name
        dme_col = [c for c in self.df.columns if 'macular' in c.lower() or 'dme' in c.lower()][0]
        dme_label = int(row[dme_col]) if not pd.isna(row[dme_col]) else 0

        return torch.tensor(tensor, dtype=torch.float32), torch.tensor(dr_label, dtype=torch.long), torch.tensor(dme_label, dtype=torch.long)

train_grade_loader = DataLoader(IDRiDGradingDataset(df_train, train_img_dir_b, is_training=True), batch_size=16, shuffle=True, num_workers=2)
test_grade_loader = DataLoader(IDRiDGradingDataset(df_test, test_img_dir_b, is_training=False), batch_size=16, shuffle=False, num_workers=2)

### Dual-Head Multi-Task Neural Network Architecture & Training

In [ ]:
class DualHeadEfficientNet(nn.Module):
    def __init__(self, num_dr_classes=5, num_dme_classes=3):
        super(DualHeadEfficientNet, self).__init__()
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        # Head 1: DR Severity Grading (0 to 4)
        self.dr_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_dr_classes)
        )

        # Head 2: Macular Edema Risk (0 to 2)
        self.dme_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_dme_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        dr_logits = self.dr_head(features)
        dme_logits = self.dme_head(features)
        return dr_logits, dme_logits

grading_model = DualHeadEfficientNet().to(device)

# Class-weighted cross entropy for imbalanced DR classes
dr_counts = np.bincount(df_train['Retinopathy grade'], minlength=5)
dr_weights = torch.tensor(1.0 / (dr_counts + 1e-5) / (1.0 / (dr_counts + 1e-5)).sum() * 5.0, dtype=torch.float32).to(device)
dr_criterion = nn.CrossEntropyLoss(weight=dr_weights)
dme_criterion = nn.CrossEntropyLoss()

grade_optimizer = optim.AdamW(grading_model.parameters(), lr=3e-4, weight_decay=1e-3)
grade_scheduler = optim.lr_scheduler.CosineAnnealingLR(grade_optimizer, T_max=20, eta_min=1e-6)
grade_scaler = torch.amp.GradScaler('cuda')

best_qwk = -1.0
print(f"\n--- [Task 2] Training Dual-Head Model on {device} (20 Epochs) ---")

for epoch in range(1, 21):
    grading_model.train()
    train_loss = 0.0
    for imgs, dr_targets, dme_targets in train_grade_loader:
        imgs, dr_targets, dme_targets = imgs.to(device), dr_targets.to(device), dme_targets.to(device)
        grade_optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            dr_out, dme_out = grading_model(imgs)
            loss = dr_criterion(dr_out, dr_targets) + 0.5 * dme_criterion(dme_out, dme_targets)
        grade_scaler.scale(loss).backward()
        grade_scaler.step(grade_optimizer)
        grade_scaler.update()
        train_loss += loss.item() * imgs.size(0)
    grade_scheduler.step()

    # Evaluation on Held-Out Test Set (103 scans)
    grading_model.eval()
    dr_preds, dr_trues = [], []
    dme_preds, dme_trues = [], []
    with torch.no_grad():
        for imgs, dr_targets, dme_targets in test_grade_loader:
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda'):
                dr_out, dme_out = grading_model(imgs)
            dr_preds.extend(torch.argmax(dr_out, dim=1).cpu().numpy())
            dr_trues.extend(dr_targets.numpy())
            dme_preds.extend(torch.argmax(dme_out, dim=1).cpu().numpy())
            dme_trues.extend(dme_targets.numpy())

    qwk = cohen_kappa_score(dr_trues, dr_preds, weights='quadratic')
    dr_acc = accuracy_score(dr_trues, dr_preds)
    dme_acc = accuracy_score(dme_trues, dme_preds)

    # Referable DR sensitivity & specificity (Grade >= 2)
    b_true = (np.array(dr_trues) >= 2).astype(int)
    b_pred = (np.array(dr_preds) >= 2).astype(int)
    tp = np.sum((b_true == 1) & (b_pred == 1))
    fn = np.sum((b_true == 1) & (b_pred == 0))
    tn = np.sum((b_true == 0) & (b_pred == 0))
    fp = np.sum((b_true == 0) & (b_pred == 1))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"Epoch [{epoch:02d}/20] | Train Loss: {train_loss/len(df_train):.4f} | DR Acc: {dr_acc*100:.1f}% | DME Acc: {dme_acc*100:.1f}% | QWK: {qwk:.4f} | Ref Sens: {sens*100:.1f}% | Spec: {spec*100:.1f}%")

    if qwk > best_qwk:
        best_qwk = qwk
        torch.save({
            'epoch': epoch,
            'model_state_dict': grading_model.state_dict(),
            'best_qwk': best_qwk,
            'backbone': 'efficientnet_b3',
            'architecture': 'dual_head_dr_dme'
        }, "/content/checkpoints/idrid_grading_efficientnet_b3.pt")
        print(f"  >>> ⭐ Saved best DR/DME model: /content/checkpoints/idrid_grading_efficientnet_b3.pt (QWK: {best_qwk:.4f})")

print(f"\n✓ [Task 2 Complete] Best Test QWK Score on IDRiD: {best_qwk:.4f}")

## 6. Visual Inspection: Fundus vs Ground Truth vs Predicted Lesions

In [ ]:
import matplotlib.pyplot as plt

unet_model.eval()
imgs, masks = next(iter(test_seg_loader))
with torch.no_grad():
    preds = torch.sigmoid(unet_model(imgs.to(device))).cpu().numpy()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

for i in range(2):
    orig_img = (imgs[i].numpy() * std + mean).transpose(1, 2, 0)
    orig_img = np.clip(orig_img, 0, 1)
    gt_mask = masks[i, 0].numpy()
    pred_mask = preds[i, 0]

    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title(f"Scan #{i+1}: Enhanced Retinal Fundus")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(gt_mask, cmap='hot')
    axes[i, 1].set_title(f"Ground Truth Lesions (MA/HE/EX/SE)")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_mask > 0.5, cmap='magma')
    axes[i, 2].set_title(f"U-Net AI Predicted Lesions")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 7. Package and Download Trained Model Artifacts
Downloads `idrid_trained_models.zip` containing:
- `unet_lesions.pt` (Multi-Lesion Segmentation Model)
- `idrid_grading_efficientnet_b3.pt` (Dual-Head DR & DME Grading Model)

In [ ]:
from google.colab import files
!zip -j idrid_trained_models.zip /content/checkpoints/unet_lesions.pt /content/checkpoints/idrid_grading_efficientnet_b3.pt
files.download('idrid_trained_models.zip')
print("🎉 Both IDRiD models packaged and downloading to your laptop!")